# TECHDUDES — QFF 2026 PS-O1

## Quantum optimization of network partitioning using QAOA

This notebook documents the problem instance, validates the final Processor D configuration, and reviews the saved experimental results. Saved tables are loaded so that opening the notebook does not automatically rerun every simulation.

**Reproducibility:** run this notebook from the project root after installing `requirements.txt`.


## 1. Environment and project paths

The notebook expects the project files and result tables to be in their documented locations.


In [22]:
from pathlib import Path
import json
import csv
import sys
import subprocess
import os

ROOT = Path("/content/TECHDUDES-QFF-2026-PSO1")

# Clone the repository only if it is not already present
if not ROOT.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/revanth2444-cmyk/TECHDUDES-QFF-2026-PSO1.git",
            str(ROOT),
        ],
        check=True,
    )

os.chdir(ROOT)
ROOT = Path.cwd()

print("Project root:", ROOT)
print("Python:", sys.version)

required_paths = [
    "qff_phase2.py",
    "data/instance.json",
    "processors/processor_A.json",
    "processors/processor_B.json",
    "processors/processor_D.json",
    "results/tables/results.csv",
    "results/tables/bell.csv",
    "results/tables/initialization_summary.csv",
]

missing = [p for p in required_paths if not (ROOT / p).exists()]
if missing:
    raise FileNotFoundError(f"Missing project files: {missing}")

print("All required paths are present.")

Project root: /content/TECHDUDES-QFF-2026-PSO1
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
All required paths are present.


## 2. Problem instance

The instance contains six vertices and nine weighted edges. The exact maximum cut is 22, obtained by exhaustive enumeration of the 64 possible bit assignments.


In [23]:
instance_path = ROOT / "data" / "instance.json"
instance = json.loads(instance_path.read_text(encoding="utf-8"))
print(json.dumps(instance, indent=2))

{
  "_note": "QFF 2026 Phase 2 Optimization Track - LOCKED official instances. Variable/asset/item index i maps to qubit i. Do not modify for the required benchmark.",
  "PS-O1": {
    "num_vertices": 6,
    "weighted_edges": [
      [
        0,
        2,
        1
      ],
      [
        0,
        3,
        4
      ],
      [
        0,
        4,
        4
      ],
      [
        1,
        4,
        3
      ],
      [
        1,
        5,
        3
      ],
      [
        2,
        3,
        5
      ],
      [
        2,
        4,
        1
      ],
      [
        3,
        4,
        4
      ],
      [
        3,
        5,
        2
      ]
    ],
    "qaoa_depth": "p=1 required; p=2 optional",
    "min_initializations": 5,
    "initialization_seeds": [
      0,
      1,
      2,
      3,
      4
    ]
  },
  "PS-O2": {
    "num_assets": 4,
    "expected_returns": [
      5,
      3,
      4,
      2
    ],
    "risk_matrix": [
      [
        4.0,
        0.5,
     

## 3. Validate Processor D

The final Processor D configuration is read from `processors/processor_D.json`. The challenge helper checks whether it satisfies the processor requirements for six logical qubits.


In [24]:
import importlib.util
import sys

module_path = ROOT / "qff_phase2.py"

spec = importlib.util.spec_from_file_location("qff_phase2", module_path)
assert spec is not None and spec.loader is not None, (
    f"Could not load module from {module_path}"
)

qff = importlib.util.module_from_spec(spec)
sys.modules["qff_phase2"] = qff
spec.loader.exec_module(qff)

processor_d_path = ROOT / "processors" / "processor_D.json"
processor_d = qff.load_processor(str(processor_d_path))

validation_errors = qff.validate_processor_d(processor_d, 6)

print("Processor D validation errors:", validation_errors)
assert not validation_errors, f"Validation failed: {validation_errors}"
print("Processor D validation passed!")

Processor D validation errors: []
Processor D validation passed!


## 4. Combined QAOA results

The results table contains Processor B and D ideal/noisy evaluations. Processor A is marked not executable because it has five physical qubits while this instance requires six logical qubits.


In [25]:
results_path = ROOT / "results" / "tables" / "results.csv"

with results_path.open("r", newline="", encoding="utf-8") as f:
    results = list(csv.DictReader(f))

print("Number of result rows:", len(results))
print()

for row in results:
    print(row)

Number of result rows: 6

{'processor': 'Processor A', 'noisy': 'False', 'noise_model': 'none', 'shots': '4096', 'seed_simulator': '42', 'exact_optimum': '22', 'best_initialization_seed': '', 'gamma': '', 'beta': '', 'mean_cut': '', 'sample_std_cut': '', 'best_sampled_cut': '', 'approx_ratio': '', 'r_best': '', 'p_opt': '', 'executable': 'False', 'logical_qubits': '6', 'physical_qubits': '5', 'logical_cx': '', 'cx': '', 'one_q_gates': '', 'depth': '', 'two_q_depth': '', 'swaps': '', 'cx_overhead': '', 'notes': 'Not executable: 5 physical qubits available, 6 logical qubits required.'}
{'processor': 'Processor A', 'noisy': 'True', 'noise_model': 'official', 'shots': '4096', 'seed_simulator': '42', 'exact_optimum': '22', 'best_initialization_seed': '', 'gamma': '', 'beta': '', 'mean_cut': '', 'sample_std_cut': '', 'best_sampled_cut': '', 'approx_ratio': '', 'r_best': '', 'p_opt': '', 'executable': 'False', 'logical_qubits': '6', 'physical_qubits': '5', 'logical_cx': '', 'cx': '', 'one_q_g

## 5. Bell-test results

Bell-test correlations and their standard errors are stored in `results/tables/bell.csv`.


In [26]:
bell_path = ROOT / "results" / "tables" / "bell.csv"

with bell_path.open("r", newline="", encoding="utf-8") as f:
    bell_results = list(csv.DictReader(f))

print("Number of Bell-test rows:", len(bell_results))
print()

for row in bell_results:
    print(row)

Number of Bell-test rows: 4

{'processor': 'Processor D', 'pair_type': 'adjacent', 'pair': '(0, 1)', 'pair_distance': '1', 'noisy': 'False', 'shots': '4096', 'XX': '1.0', 'XX_se': '0.0', 'YY': '-1.0', 'YY_se': '0.0', 'ZZ': '1.0', 'ZZ_se': '0.0', 'fidelity': '1.0', 'fidelity_se': '0.0', 'cx': '1'}
{'processor': 'Processor D', 'pair_type': 'adjacent', 'pair': '(0, 1)', 'pair_distance': '1', 'noisy': 'True', 'shots': '4096', 'XX': '0.94580078125', 'XX_se': '0.005074178326602111', 'YY': '-0.9521484375', 'YY_se': '0.004775562205284335', 'ZZ': '0.955078125', 'ZZ_se': '0.004630533407219104', 'fidelity': '0.9632568359375', 'fidelity_se': '0.0020915771516679545', 'cx': '1'}
{'processor': 'Processor D', 'pair_type': 'farthest', 'pair': '(0, 8)', 'pair_distance': '4', 'noisy': 'False', 'shots': '4096', 'XX': '1.0', 'XX_se': '0.0', 'YY': '-1.0', 'YY_se': '0.0', 'ZZ': '1.0', 'ZZ_se': '0.0', 'fidelity': '1.0', 'fidelity_se': '0.0', 'cx': '10'}
{'processor': 'Processor D', 'pair_type': 'farthest', 'p

## 6. Initialization summary

Five initializations were tested with seeds 0 through 4. This summary records their mean performance and variation.


In [27]:
summary_path = ROOT / "results" / "tables" / "initialization_summary.csv"

with summary_path.open("r", newline="", encoding="utf-8") as f:
    initialization_results = list(csv.DictReader(f))

print("Number of initialization-summary rows:", len(initialization_results))
print()

for row in initialization_results:
    print(row)

Number of initialization-summary rows: 1

{'initialization_count': '5', 'seeds': '0;1;2;3;4', 'mean_ideal_expected_cut': '15.525749902292258', 'mean_approx_ratio': '0.7057159046496482', 'population_std_approx_ratio': '0.0012509368330797207', 'sample_std_approx_ratio': '0.0013985898971622813', 'best_seed': '2', 'best_approx_ratio': '0.7071628760115108'}


## 7. Interpretation and limitations

In these simulations, both Processor B and Processor D sampled the optimal cut of 22 in their ideal and noisy runs. Their optimal-cut probabilities were below 0.08, so most individual measurement samples were not optimal.

Processor D used more transpiled CX gates than Processor B in the recorded circuits. The noisy farthest-pair Bell test also had lower fidelity than the adjacent-pair test. These are results for the configured simulator and noise model; they are not physical-hardware measurements.

See `report.md` for the report, detailed result tables, resource comparison, and AI-use disclosure.
